# MP4 から before / after 候補を選ぶ

この notebook は `analysis/run_video_roi_search.py` を操作するための薄い入口です。
肌色・Lab値では候補を順位付けせず、顔向き・顔サイズ・ROI形状・目の開き・口の開きなどの幾何条件で before / after 候補を探します。

ここでは候補抽出と目視確認までを行います。Lab解析やメイク採点は自動では実行しません。


In [ ]:
from pathlib import Path

# ---- 実験条件 ----
VIDEO = Path('makeup2.mp4')
OUTPUT = Path('outputs/makeup_video_search_notebook')

# 秒。動画内で「メイク前」「完成後」と確認できた区間だけを指定する。
BEFORE_RANGE = (65.0, 196.0)
AFTER_RANGE = (2022.0, 2124.0)

INTERVAL = 5.0
REFINE_INTERVAL = 1.0
TOP = 10

# 目視確認前は None のまま。候補1を採用するなら後で 1 に変更する。
APPROVED_RANK = None


In [ ]:
import math
import shutil

# fail-fast: 実行場所・入力・外部コマンドを先に検証する。
if not Path('analysis/run_video_roi_search.py').is_file():
    raise RuntimeError('リポジトリ直下で notebook を実行してください。analysis/run_video_roi_search.py が見つかりません。')
if not VIDEO.is_file():
    raise FileNotFoundError(f'動画が見つかりません: {VIDEO}')
if shutil.which('ffprobe') is None:
    raise RuntimeError('ffprobe が PATH にありません。動画長を安全に取得できないため停止します。')
if not all(math.isfinite(v) for v in (*BEFORE_RANGE, *AFTER_RANGE, INTERVAL, REFINE_INTERVAL)):
    raise ValueError('時刻・間隔は有限値で指定してください。')
if not (0 <= BEFORE_RANGE[0] < BEFORE_RANGE[1] < AFTER_RANGE[0] < AFTER_RANGE[1]):
    raise ValueError('BEFORE_RANGE と AFTER_RANGE は重ならない昇順区間で指定してください。')
if INTERVAL <= 0 or REFINE_INTERVAL <= 0 or REFINE_INTERVAL > INTERVAL:
    raise ValueError('0 < REFINE_INTERVAL <= INTERVAL を満たしてください。')
if TOP < 1:
    raise ValueError('TOP は1以上にしてください。')

print('preflight OK')

## 候補探索を実行

`run_video_roi_search.py` と同じ処理を呼びます。粗探索は `BEFORE_RANGE` と `AFTER_RANGE` の中だけを走査し、動画の途中の未使用区間は解析しません。
既存の非空出力フォルダには上書きしません。
途中のROI生成に失敗したフレームは理由を記録して候補から除外しますが、入力形式や処理方法を別方式へ切り替えるフォールバックはしません。


In [ ]:
from analysis.run_video_roi_search import parse_args, run

argv = [
    '--video', str(VIDEO),
    '--output', str(OUTPUT),
    '--before-range', str(BEFORE_RANGE[0]), str(BEFORE_RANGE[1]),
    '--after-range', str(AFTER_RANGE[0]), str(AFTER_RANGE[1]),
    '--interval', str(INTERVAL),
    '--refine-interval', str(REFINE_INTERVAL),
    '--top', str(TOP),
]

manifest = run(parse_args(argv))
if manifest['status'] != 'needs_review':
    raise RuntimeError(f"比較候補を作れませんでした: status={manifest['status']}")

print('候補生成完了:', OUTPUT / 'report.html')

## 最良候補を表示

ここで必ず目視します。頬・額ROI、髪・手・影・道具、顔向き、表情、before/afterの意味が妥当かを確認してください。


In [ ]:
from IPython.display import display, Image

for name in ('best_pair_faces.png', 'best_pair_roi_overlay.png'):
    path = OUTPUT / name
    if not path.is_file():
        raise FileNotFoundError(f'レビュー画像がありません: {path}')
    display(Image(filename=str(path)))

## 上位候補の数値を確認

スコアは「幾何的な差」で、小さいほど条件が近い候補です。美しさ・メイク効果・本人一致の確率ではありません。


In [ ]:
import json
from IPython.display import HTML, display

matching_path = OUTPUT / 'matching.json'
if not matching_path.is_file():
    raise FileNotFoundError(matching_path)

matching = json.loads(matching_path.read_text(encoding='utf-8'))
pairs = matching.get('ranked_pairs', [])
if not pairs:
    raise RuntimeError('ranked_pairs が空です。')

columns = [
    ('rank', 'rank'), ('before_time', 'before_time'), ('after_time', 'after_time'),
    ('score', 'score'), ('yaw_gap', 'yaw_gap_degrees'), ('pitch_gap', 'pitch_gap_degrees'),
    ('roll_gap', 'roll_gap_degrees'), ('face_scale_ratio', 'face_scale_ratio'),
    ('roi_rms', 'roi_procrustes_rms'), ('eye_gap', 'eye_aperture_gap'), ('mouth_gap', 'mouth_opening_gap'),
]

headers = ''.join(f'<th>{label}</th>' for label, _ in columns)
body = []
for rank, pair in enumerate(pairs, 1):
    terms = pair['terms']
    values = {
        'rank': rank,
        'before_time': pair['before_time'],
        'after_time': pair['after_time'],
        'score': pair['score'],
        **terms,
    }
    cells = ''.join(f'<td>{values[key]:.4f}</td>' if isinstance(values[key], float) else f'<td>{values[key]}</td>' for _, key in columns)
    body.append(f'<tr>{cells}</tr>')

display(HTML(f'<table><thead><tr>{headers}</tr></thead><tbody>{"".join(body)}</tbody></table>'))

## 目視承認したペアを固定

上の設定セルで `APPROVED_RANK = 1` のように変更してから実行します。`None` のままなら停止します。
このセルはLab解析をしません。採用したフレームID・時刻・元画像・ROIディレクトリ・SHA-256を `selected_pair.json` に固定します。


In [ ]:
import hashlib

if APPROVED_RANK is None:
    raise RuntimeError('目視確認後に APPROVED_RANK を設定してください。自動承認はしません。')
if isinstance(APPROVED_RANK, bool) or not isinstance(APPROVED_RANK, int):
    raise TypeError('APPROVED_RANK は整数で指定してください。')
if not (1 <= APPROVED_RANK <= len(pairs)):
    raise ValueError(f'APPROVED_RANK は 1..{len(pairs)} の範囲です。')

pair = pairs[APPROVED_RANK - 1]
scan_manifest = json.loads((OUTPUT / 'scan_manifest.json').read_text(encoding='utf-8'))
records = {record['frame_id']: record for record in scan_manifest['records']}

def sha256(path: Path) -> str:
    if not path.is_file():
        raise FileNotFoundError(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

selected = {'rank': APPROVED_RANK, 'score': pair['score'], 'before': {}, 'after': {}}
for phase, key in (('before', 'before_id'), ('after', 'after_id')):
    record = records.get(pair[key])
    if record is None:
        raise RuntimeError(f"matching.json の {phase} フレームが scan_manifest.json にありません: {pair[key]}")
    image_path = Path(record['image_path'])
    roi_dir = Path(record['roi_dir'])
    masks_path = roi_dir / 'roi_masks.npz'
    points_path = roi_dir / 'roi_points.json'
    overlay_path = roi_dir / 'roi_overlay.png'
    for required in (image_path, masks_path, points_path, overlay_path):
        if not required.is_file():
            raise FileNotFoundError(required)
    selected[phase] = {
        'frame_id': record['frame_id'],
        'timestamp_seconds': record['timestamp_seconds'],
        'image_path': str(image_path),
        'image_sha256': sha256(image_path),
        'roi_dir': str(roi_dir),
        'roi_masks_sha256': sha256(masks_path),
        'roi_points_sha256': sha256(points_path),
        'roi_overlay_sha256': sha256(overlay_path),
    }

selected_path = OUTPUT / 'selected_pair.json'
if selected_path.exists():
    raise FileExistsError(f'既に選択結果があります。上書きしません: {selected_path}')
selected_path.write_text(json.dumps(selected, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('固定しました:', selected_path)
selected

## 次の段階

`selected_pair.json` で採用ペアを固定した後、既存のLab解析へ接続します。
動画候補探索とLab解析を一度に自動実行せず、目視承認を境界として分けておきます。
